# VD-EXP-002+ - BDD100K End-to-End Vehicle Detection Pipeline

Bu notebook, Anomali Road Safety AI için BDD100K tabanlı araç tespit hattını tek dosyada yürütür:

1. BDD100K indirme veya mevcut Drive verisini doğrulama
2. Drive altında doğru klasör yapısına yerleştirme
3. BDD100K JSON annotation -> YOLO label dönüşümü
4. Condition metadata koruma (`weather`, `timeofday`, `scene`)
5. Pretrained baseline validation
6. Fine-tune training
7. Fine-tuned validation
8. Condition breakdown validation
9. Baseline vs fine-tuned fark tablosu
10. Export

Ana hedef `vehicle_detector_general` modelidir. Bu model yalnız gündüz/normal koşul modeli değildir; `night_low_light`, `rain`, `fog_low_visibility` gibi koşullar validation kırılımında korunur. Specialist detector bu notebook'ta eğitilmez; specialist kararı bu notebook'un metriklerinden sonra verilir.

Ham veri, model ağırlıkları ve run çıktıları Git'e eklenmez; Google Drive altında tutulur.

## 0. Kullanım Özeti

Notebook tek başına çalışacak şekilde tasarlanmıştır. Önce aşağıdaki `Config` hücresindeki indirme modunu seçin.

Desteklenen indirme modları:

* `manual`: BDD100K daha önce Drive'a konmuşsa sadece path doğrular.
* `kaggle`: Kaggle dataset mirror/slug üzerinden indirir.
* `direct`: Kullanıcı tarafından verilen doğrudan arşiv URL'lerini indirir.
* `gdown`: Google Drive file ID/URL ile indirir.

Önerilen Drive hedefi:

```text
/content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k/
```

Beklenen veri yapısı, indirilen sürüme göre şu adaylardan biri olabilir:

```text
bdd100k/
  images/100k/train/
  images/100k/val/
  labels/det_20/det_train.json
  labels/det_20/det_val.json
```

veya:

```text
bdd100k/
  images/train/
  images/val/
  labels/bdd100k_labels_images_train.json
  labels/bdd100k_labels_images_val.json
```

Notebook yaygın path adaylarını otomatik arar. Gerekirse arşiv açıldıktan sonra klasörleri bu düzene taşıyın.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

!pip install -q ultralytics pyyaml pandas pillow tqdm kaggle gdown

from pathlib import Path
from urllib.parse import urlparse
from urllib.request import urlretrieve
import getpass
import json
import os
import shutil
import subprocess
import sys
import tarfile
import zipfile

import pandas as pd
import yaml
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

print('Colab environment ready')

## 1. Config

Aşağıdaki ayarlar notebook'un tamamını kontrol eder.

İlk çalışma için öneri:

* `DOWNLOAD_METHOD = 'kaggle'`, `direct` veya `gdown` kullanacaksan ilgili alanları doldur.
* Sadece YOLO11n koşmak için `RUN_YOLO11N=True`, diğer challenger'ları `False` bırak.
* İlk smoke run için `EPOCHS=5`; gerçek deney için `30+` yapılabilir.

Kaggle kullanacaksan Colab secrets/env içinde `KAGGLE_USERNAME` ve `KAGGLE_KEY` tanımlı olmalı veya `kaggle.json` yüklenmeli.

In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/anomali-road-safety-ai')
BDD_ROOT = PROJECT_ROOT / 'datasets' / 'bdd100k'
OUT_ROOT = PROJECT_ROOT / 'datasets' / 'bdd100k_vehicle_yolo'
RUN_ROOT = PROJECT_ROOT / 'runs' / 'bdd100k_vehicle_detection'
EXPORT_ROOT = PROJECT_ROOT / 'exports' / 'bdd100k_vehicle_detection'

# Download config. Use 'manual', 'kaggle', 'direct', or 'gdown'.
DOWNLOAD_METHOD = 'manual'
EXTRACT_DOWNLOADS = True

# Kaggle example: 'owner/dataset-slug'. Keep blank until selected.
KAGGLE_DATASET = ''

# Direct archive URLs. Keep secrets/tokens out of Git; fill inside Colab runtime.
DIRECT_URLS = []

# Google Drive file IDs or URLs for gdown.
GDRIVE_IDS_OR_URLS = []

# Training config
IMG_SIZE = 640
EPOCHS = 30
BATCH = 16
WORKERS = 2
SEED = 42
PATIENCE = 10

# Enable only the runs you want. All share the same converted BDD100K split.
EXPERIMENTS = [
    {
        'experiment_id': 'VD-EXP-002',
        'model_name': 'yolo11n.pt',
        'family': 'YOLO11',
        'variant': 'yolo11n',
        'enabled': True,
        'train': True,
        'export': True,
    },
    {
        'experiment_id': 'VD-EXP-003',
        'model_name': 'yolo11s.pt',
        'family': 'YOLO11',
        'variant': 'yolo11s',
        'enabled': False,
        'train': True,
        'export': False,
    },
    {
        'experiment_id': 'VD-EXP-004',
        'model_name': 'yolov10n.pt',
        'family': 'YOLOv10',
        'variant': 'yolov10n',
        'enabled': False,
        'train': True,
        'export': False,
    },
]

TARGET_CLASSES = {
    'car': 0,
    'bus': 1,
    'truck': 2,
    'motor': 3,
    'motorcycle': 3,
    'motorbike': 3,
}
NAMES = ['car', 'bus', 'truck', 'motorcycle']

for p in [BDD_ROOT, OUT_ROOT, RUN_ROOT, EXPORT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('BDD_ROOT:', BDD_ROOT)
print('OUT_ROOT:', OUT_ROOT)
print('RUN_ROOT:', RUN_ROOT)
print('Enabled experiments:', [e['experiment_id'] for e in EXPERIMENTS if e['enabled']])

## 2. Kaggle Credential Setup

Bu hücre yalnız `DOWNLOAD_METHOD = 'kaggle'` ise çalışır.

Credential sırası:

1. Colab Secrets içinde `KAGGLE_USERNAME` ve `KAGGLE_KEY` varsa otomatik okunur.
2. Yoksa runtime sırasında username ve API key istenir.
3. API key ekrana yazdırılmaz ve notebook/repo içine kaydedilmez.

Bu projede API key düz metin olarak Git'e veya notebook hücresine yazılmamalıdır.

In [ ]:
def setup_kaggle_credentials_if_needed():
    if DOWNLOAD_METHOD != 'kaggle':
        print('Kaggle credential setup skipped; DOWNLOAD_METHOD != kaggle')
        return

    username = os.getenv('KAGGLE_USERNAME')
    key = os.getenv('KAGGLE_KEY')

    if not username or not key:
        try:
            from google.colab import userdata
            username = username or userdata.get('KAGGLE_USERNAME')
            key = key or userdata.get('KAGGLE_KEY')
        except Exception:
            pass

    if not username:
        username = input('Kaggle username: ').strip()
    if not key:
        key = getpass.getpass('Kaggle API key: ').strip()

    if not username or not key:
        raise RuntimeError('Kaggle credentials are required for DOWNLOAD_METHOD=kaggle')

    os.environ['KAGGLE_USERNAME'] = username
    os.environ['KAGGLE_KEY'] = key
    print('Kaggle credentials loaded into runtime environment. Key was not printed or saved.')


setup_kaggle_credentials_if_needed()

## 3. BDD100K Download / Placement

Bu hücre seçilen moda göre BDD100K'i Drive altına indirir veya mevcut veriyi doğrular.

Önemli: Official portal veya mirror yapısı değişebilir. İndirme sonrası notebook path kontrolü yapar; eksik path varsa arşivden çıkan klasörleri `BDD_ROOT` altında beklenen düzene taşımanız gerekir.

In [ ]:
def run(cmd):
    print('+', ' '.join(map(str, cmd)))
    subprocess.check_call(list(map(str, cmd)))


def extract_archive(archive_path, output_dir):
    archive_path = Path(archive_path)
    output_dir = Path(output_dir)
    suffixes = ''.join(archive_path.suffixes).lower()
    print('Extracting:', archive_path)
    if archive_path.suffix.lower() == '.zip':
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(output_dir)
    elif suffixes.endswith('.tar.gz') or suffixes.endswith('.tgz'):
        with tarfile.open(archive_path, 'r:gz') as tf:
            tf.extractall(output_dir)
    elif archive_path.suffix.lower() == '.tar':
        with tarfile.open(archive_path, 'r') as tf:
            tf.extractall(output_dir)
    else:
        print('Unknown archive type, kept as file:', archive_path)


def download_direct(urls, output_dir, extract=True):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for url in urls:
        name = Path(urlparse(url).path).name or 'bdd100k_download'
        target = output_dir / name
        if not target.exists():
            print('Downloading:', url)
            urlretrieve(url, target)
        else:
            print('Exists, skipping:', target)
        if extract:
            extract_archive(target, output_dir)


def download_gdown(ids_or_urls, output_dir, extract=True):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for item in ids_or_urls:
        run(['gdown', '--fuzzy', item, '-O', str(output_dir)])
    if extract:
        for archive in output_dir.iterdir():
            suffixes = ''.join(archive.suffixes).lower()
            if archive.is_file() and (archive.suffix.lower() in {'.zip', '.tar'} or suffixes.endswith(('.tar.gz', '.tgz'))):
                extract_archive(archive, output_dir)


def download_kaggle(dataset_slug, output_dir, extract=True):
    if not dataset_slug:
        raise ValueError('KAGGLE_DATASET is empty. Set it to owner/dataset-slug.')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    if not kaggle_json.exists() and not (os.getenv('KAGGLE_USERNAME') and os.getenv('KAGGLE_KEY')):
        raise RuntimeError('Kaggle credentials missing. Set KAGGLE_USERNAME and KAGGLE_KEY in Colab secrets/env or upload kaggle.json.')
    cmd = ['kaggle', 'datasets', 'download', '-d', dataset_slug, '-p', str(output_dir)]
    if extract:
        cmd.append('--unzip')
    run(cmd)


def structure_status(root):
    root = Path(root)
    candidates = {
        'train_images': [root / 'images' / '100k' / 'train', root / 'images' / 'train'],
        'val_images': [root / 'images' / '100k' / 'val', root / 'images' / 'val'],
        'train_labels': [
            root / 'labels' / 'det_20' / 'det_train.json',
            root / 'labels' / 'bdd100k_labels_images_train.json',
            root / 'bdd100k_labels_images_train.json',
        ],
        'val_labels': [
            root / 'labels' / 'det_20' / 'det_val.json',
            root / 'labels' / 'bdd100k_labels_images_val.json',
            root / 'bdd100k_labels_images_val.json',
        ],
    }
    status = {key: any(p.exists() for p in paths) for key, paths in candidates.items()}
    for key, ok in status.items():
        print(f'{key}: {"OK" if ok else "MISSING"}')
    return status


if DOWNLOAD_METHOD == 'manual':
    print('Manual mode: skipping download, checking existing BDD_ROOT')
elif DOWNLOAD_METHOD == 'kaggle':
    download_kaggle(KAGGLE_DATASET, BDD_ROOT, EXTRACT_DOWNLOADS)
elif DOWNLOAD_METHOD == 'direct':
    download_direct(DIRECT_URLS, BDD_ROOT, EXTRACT_DOWNLOADS)
elif DOWNLOAD_METHOD == 'gdown':
    download_gdown(GDRIVE_IDS_OR_URLS, BDD_ROOT, EXTRACT_DOWNLOADS)
else:
    raise ValueError(f'Unknown DOWNLOAD_METHOD: {DOWNLOAD_METHOD}')

status = structure_status(BDD_ROOT)
if not all(status.values()):
    print('
Expected structure not fully found yet. Inspect extracted files under:', BDD_ROOT)
    print('You can move folders inside Drive, then rerun this cell.')

## 4. BDD100K Path Discovery

Bu bölüm BDD100K image/label path'lerini bulur. Path bulunamazsa önce indirme/yerleşim adımını düzeltin.

In [ ]:
def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError('No candidate path found:
' + '
'.join(str(p) for p in paths))

TRAIN_LABEL = first_existing([
    BDD_ROOT / 'labels' / 'det_20' / 'det_train.json',
    BDD_ROOT / 'labels' / 'bdd100k_labels_images_train.json',
    BDD_ROOT / 'bdd100k_labels_images_train.json',
])
VAL_LABEL = first_existing([
    BDD_ROOT / 'labels' / 'det_20' / 'det_val.json',
    BDD_ROOT / 'labels' / 'bdd100k_labels_images_val.json',
    BDD_ROOT / 'bdd100k_labels_images_val.json',
])
TRAIN_IMG_DIR = first_existing([
    BDD_ROOT / 'images' / '100k' / 'train',
    BDD_ROOT / 'images' / 'train',
])
VAL_IMG_DIR = first_existing([
    BDD_ROOT / 'images' / '100k' / 'val',
    BDD_ROOT / 'images' / 'val',
])

print('TRAIN_LABEL:', TRAIN_LABEL)
print('VAL_LABEL:', VAL_LABEL)
print('TRAIN_IMG_DIR:', TRAIN_IMG_DIR)
print('VAL_IMG_DIR:', VAL_IMG_DIR)

## 5. Condition Mapping

Condition metadata araç sınıfı değildir. Sadece balancing, validation breakdown ve future specialist kararı için tutulur.

In [ ]:
def normalize_text(value):
    if value is None:
        return 'undefined'
    return str(value).strip().lower()


def condition_from_attributes(attrs):
    attrs = attrs or {}
    weather = normalize_text(attrs.get('weather'))
    timeofday = normalize_text(attrs.get('timeofday'))
    scene = normalize_text(attrs.get('scene'))

    if weather in {'foggy', 'fog'}:
        primary = 'fog_low_visibility'
    elif weather in {'rainy', 'rain'}:
        primary = 'rain'
    elif timeofday == 'night':
        primary = 'night_low_light'
    elif timeofday in {'dawn/dusk', 'dawn', 'dusk'}:
        primary = 'low_light_transition'
    elif weather in {'snowy', 'snow', 'sandstorm'}:
        primary = 'adverse_other'
    elif weather in {'clear', 'overcast', 'partly cloudy'} or timeofday in {'daytime', 'day'}:
        primary = 'day_clear'
    else:
        primary = 'unknown'

    return {
        'condition_profile': primary,
        'weather': weather,
        'timeofday': timeofday,
        'scene': scene,
    }

condition_from_attributes({'weather': 'rainy', 'timeofday': 'night', 'scene': 'city street'})

## 6. BDD JSON -> YOLO Label Conversion

Bu bölüm `car`, `bus`, `truck`, `motorcycle` sınıflarını çıkarır ve YOLO label formatına dönüştürür. Image path'leri YOLO formatlı çalışma klasöründe symlink olarak tutulur; symlink desteklenmezse kopyalama fallback çalışır.

In [ ]:
def box2d_to_yolo(box, img_w, img_h):
    x1 = max(0.0, float(box['x1']))
    y1 = max(0.0, float(box['y1']))
    x2 = min(float(img_w), float(box['x2']))
    y2 = min(float(img_h), float(box['y2']))
    if x2 <= x1 or y2 <= y1:
        return None
    xc = ((x1 + x2) / 2.0) / img_w
    yc = ((y1 + y2) / 2.0) / img_h
    bw = (x2 - x1) / img_w
    bh = (y2 - y1) / img_h
    return xc, yc, bw, bh


def link_or_copy(src, dst):
    src = Path(src)
    dst = Path(dst)
    if dst.exists():
        return
    try:
        dst.symlink_to(src)
    except Exception:
        shutil.copy2(src, dst)


def convert_split(split_name, label_json, img_dir):
    out_img_dir = OUT_ROOT / 'images' / split_name
    out_lbl_dir = OUT_ROOT / 'labels' / split_name
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    records = json.loads(Path(label_json).read_text())
    meta_rows = []
    image_list = []
    kept_boxes = 0
    skipped_images = 0
    empty_vehicle_images = 0

    for item in tqdm(records, desc=f'convert {split_name}'):
        name = item.get('name')
        src_img = Path(img_dir) / name
        if not src_img.exists():
            skipped_images += 1
            continue

        with Image.open(src_img) as im:
            img_w, img_h = im.size

        lines = []
        for obj in item.get('labels', []) or []:
            category = normalize_text(obj.get('category'))
            if category not in TARGET_CLASSES:
                continue
            box = obj.get('box2d')
            if not box:
                continue
            converted = box2d_to_yolo(box, img_w, img_h)
            if converted is None:
                continue
            cls_id = TARGET_CLASSES[category]
            xc, yc, bw, bh = converted
            lines.append(f'{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')

        if not lines:
            empty_vehicle_images += 1
            continue

        dst_img = out_img_dir / name
        link_or_copy(src_img, dst_img)
        (out_lbl_dir / f'{Path(name).stem}.txt').write_text('
'.join(lines) + '
')

        cond = condition_from_attributes(item.get('attributes'))
        image_path = str(dst_img)
        image_list.append(image_path)
        meta_rows.append({
            'split': split_name,
            'image': image_path,
            'name': name,
            'vehicle_box_count': len(lines),
            **cond,
        })
        kept_boxes += len(lines)

    return meta_rows, image_list, kept_boxes, skipped_images, empty_vehicle_images

train_meta, train_images, train_boxes, train_skipped, train_empty = convert_split('train', TRAIN_LABEL, TRAIN_IMG_DIR)
val_meta, val_images, val_boxes, val_skipped, val_empty = convert_split('val', VAL_LABEL, VAL_IMG_DIR)

metadata = pd.DataFrame(train_meta + val_meta)
(OUT_ROOT / 'metadata').mkdir(exist_ok=True, parents=True)
metadata.to_csv(OUT_ROOT / 'metadata' / 'condition_metadata.csv', index=False)

print('train images:', len(train_images), 'train boxes:', train_boxes, 'missing images:', train_skipped, 'empty vehicle images:', train_empty)
print('val images:', len(val_images), 'val boxes:', val_boxes, 'missing images:', val_skipped, 'empty vehicle images:', val_empty)
print(metadata.groupby(['split', 'condition_profile']).size())

## 7. YOLO `data.yaml` ve Condition Validation Listeleri

In [ ]:
splits_dir = OUT_ROOT / 'splits'
splits_dir.mkdir(parents=True, exist_ok=True)

(splits_dir / 'train.txt').write_text('
'.join(train_images) + '
')
(splits_dir / 'val.txt').write_text('
'.join(val_images) + '
')

data_yaml = {
    'path': str(OUT_ROOT),
    'train': str(splits_dir / 'train.txt'),
    'val': str(splits_dir / 'val.txt'),
    'names': {i: name for i, name in enumerate(NAMES)},
}
(OUT_ROOT / 'data.yaml').write_text(yaml.safe_dump(data_yaml, sort_keys=False))

condition_yamls = {}
val_df = metadata[metadata['split'] == 'val'].copy()
for condition, group in val_df.groupby('condition_profile'):
    condition_list = splits_dir / f'val_{condition}.txt'
    condition_list.write_text('
'.join(group['image'].tolist()) + '
')
    condition_yaml = OUT_ROOT / f'data_val_{condition}.yaml'
    condition_yaml.write_text(yaml.safe_dump({
        'path': str(OUT_ROOT),
        'train': str(splits_dir / 'train.txt'),
        'val': str(condition_list),
        'names': {i: name for i, name in enumerate(NAMES)},
    }, sort_keys=False))
    condition_yamls[condition] = condition_yaml

print('data.yaml:', OUT_ROOT / 'data.yaml')
print('condition yamls:', condition_yamls)

## 8. Metrik Yardımcıları

In [ ]:
def metrics_to_row(metrics, prefix=''):
    return {
        f'{prefix}map50': float(metrics.box.map50),
        f'{prefix}map5095': float(metrics.box.map),
        f'{prefix}precision_mean': float(metrics.box.mp),
        f'{prefix}recall_mean': float(metrics.box.mr),
    }


def validate_overall(model, run_dir, name):
    return model.val(
        data=str(OUT_ROOT / 'data.yaml'),
        imgsz=IMG_SIZE,
        project=str(run_dir),
        name=name,
        exist_ok=True,
    )


def validate_conditions(model, run_dir, experiment_id, model_label, stage):
    rows = []
    for condition, condition_yaml in condition_yamls.items():
        print(f'Validating {experiment_id} {stage} condition:', condition)
        metrics = model.val(
            data=str(condition_yaml),
            imgsz=IMG_SIZE,
            project=str(run_dir),
            name=f'val_{stage}_{condition}',
            exist_ok=True,
        )
        rows.append({
            'experiment_id': experiment_id,
            'model': model_label,
            'stage': stage,
            'condition_profile': condition,
            'val_image_count': int((metadata['split'].eq('val') & metadata['condition_profile'].eq(condition)).sum()),
            **metrics_to_row(metrics),
        })
    return rows


def add_delta(finetuned_row, baseline_row):
    result = dict(finetuned_row)
    for key in ['map50', 'map5095', 'precision_mean', 'recall_mean']:
        result[f'delta_{key}'] = result[key] - baseline_row[key]
    return result

## 9. End-to-End Experiment Loop

Bu döngü etkin olan her model için şunları yapar:

1. Pretrained baseline validation
2. Fine-tune training
3. Fine-tuned overall validation
4. Condition breakdown validation
5. Baseline vs fine-tuned delta tablosu
6. Optional export

İlk çalışmada yalnız `VD-EXP-002 / YOLO11n` etkin bırakılabilir. Sonra aynı split üzerinde `YOLO11s` ve `YOLOv10n` açılır.

In [ ]:
all_overall_rows = []
all_condition_rows = []
all_delta_rows = []

for exp in EXPERIMENTS:
    if not exp.get('enabled', False):
        print('Skipping disabled experiment:', exp['experiment_id'])
        continue

    experiment_id = exp['experiment_id']
    model_name = exp['model_name']
    model_label = exp['variant']
    exp_run_dir = RUN_ROOT / f'{experiment_id}-{model_label}'
    exp_export_dir = EXPORT_ROOT / f'{experiment_id}-{model_label}'
    exp_run_dir.mkdir(parents=True, exist_ok=True)
    exp_export_dir.mkdir(parents=True, exist_ok=True)

    print('
===', experiment_id, model_label, '===')

    pretrained = YOLO(model_name)
    baseline_metrics = validate_overall(pretrained, exp_run_dir, 'val_pretrained_baseline')
    baseline_row = {
        'experiment_id': experiment_id,
        'model': model_label,
        'stage': 'pretrained_baseline',
        **metrics_to_row(baseline_metrics),
    }
    all_overall_rows.append(baseline_row)
    all_condition_rows.extend(validate_conditions(pretrained, exp_run_dir, experiment_id, model_label, 'pretrained_baseline'))

    if exp.get('train', True):
        model = YOLO(model_name)
        model.train(
            data=str(OUT_ROOT / 'data.yaml'),
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH,
            workers=WORKERS,
            seed=SEED,
            patience=PATIENCE,
            project=str(exp_run_dir),
            name='train',
            exist_ok=True,
        )
        best_weights = exp_run_dir / 'train' / 'weights' / 'best.pt'
        trained = YOLO(str(best_weights))
    else:
        trained = pretrained
        best_weights = None

    fine_metrics = validate_overall(trained, exp_run_dir, 'val_finetuned')
    fine_row = {
        'experiment_id': experiment_id,
        'model': model_label,
        'stage': 'finetuned',
        **metrics_to_row(fine_metrics),
    }
    all_overall_rows.append(fine_row)
    all_delta_rows.append(add_delta(fine_row, baseline_row))
    all_condition_rows.extend(validate_conditions(trained, exp_run_dir, experiment_id, model_label, 'finetuned'))

    if exp.get('export', False):
        exported = trained.export(format='onnx', imgsz=IMG_SIZE, dynamic=True)
        if best_weights and Path(best_weights).exists():
            shutil.copy2(best_weights, exp_export_dir / 'best.pt')
        if Path(exported).exists():
            shutil.copy2(exported, exp_export_dir / Path(exported).name)
        print('Exported to:', exp_export_dir)

summary_dir = RUN_ROOT / 'summary'
summary_dir.mkdir(parents=True, exist_ok=True)

overall_df = pd.DataFrame(all_overall_rows)
condition_df = pd.DataFrame(all_condition_rows)
delta_df = pd.DataFrame(all_delta_rows)

overall_df.to_csv(summary_dir / 'overall_metrics.csv', index=False)
condition_df.to_csv(summary_dir / 'condition_breakdown_metrics.csv', index=False)
delta_df.to_csv(summary_dir / 'baseline_vs_finetuned_delta.csv', index=False)

print('Summary dir:', summary_dir)
display(overall_df)
display(delta_df)
display(condition_df.sort_values(['experiment_id', 'stage', 'condition_profile']).head(100))

## 10. Model Seçim Yorumu

Aşağıdaki hücre condition kırılımlarına göre zayıf alanları listeler. Specialist detector kararını bu tabloya göre vereceğiz.

Kural:

* General detector overall iyi ama `night_low_light` zayıfsa ilk specialist adayı `vehicle_detector_night_low_light` olur.
* `rain` zayıfsa rain specialist daha sonra açılır.
* `fog_low_visibility` yeterli veriyle zayıfsa fog specialist açılır.
* Specialist açmadan önce `best_general` seçilir.

In [ ]:
if not condition_df.empty:
    fine_condition = condition_df[condition_df['stage'] == 'finetuned'].copy()
    weak = fine_condition.sort_values('map5095').head(10)
    print('Lowest condition breakdown rows by mAP@0.5:0.95')
    display(weak)

if not delta_df.empty:
    print('Fine-tuned uplift over pretrained baseline')
    display(delta_df.sort_values('delta_map5095', ascending=False))

## 11. Repo'ya Aktarılacak Özet

Notebook tamamlandıktan sonra yalnız küçük özet dosyaları repoya aktarılabilir:

* `overall_metrics.csv` özetinden benchmark satırı
* `baseline_vs_finetuned_delta.csv` içinden delta değerleri
* `condition_breakdown_metrics.csv` içinden condition kırılım özeti
* model card notları

Repo'ya eklenmeyecekler:

* BDD100K raw images
* YOLO labels çıktıları
* `.pt`, `.onnx`, `.tflite` ağırlıkları
* `runs/` görselleri ve büyük artefactler

Sonraki gerçek adım: MacBook local edge runtime benchmark.